# Musahhih F1-P1: guarded natural-data QLoRA pilot

This notebook prepares the frozen **F1-P1** natural-data pilot for student researchers. Its primary free runtime is a Kaggle Notebook with an NVIDIA GPU; CPU is not a training fallback. It uses Gemma 3 4B Instruct, Unsloth, 4-bit QLoRA, and private QALB-2014 L1 records. **QALB test and Nahw-Passage are never loaded here.** The notebook does not generate synthetic data and does not modify the base weights.

The workflow has two deliberate gates: first a one-batch GPU memory smoke test, then the complete two-epoch run. Full training is disabled by default. QALB text, prompts, predictions, adapters, and logs remain private and Git-ignored.

## 1. Runtime and GPU verification
In Kaggle open **Settings**, enable a **GPU accelerator** and **Internet**, then restart the session if needed. The exact GPU is recorded rather than assumed. Free accelerator availability is not guaranteed; no paid tier is required.

In [ ]:
import subprocess, sys
assigned_gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip()
P100_STACK = {'torch': '2.6.0', 'torchvision': '0.21.0', 'xformers': '0.0.29.post3', 'torchao': '0.12.0', 'numpy': '2.0.2', 'index': 'https://download.pytorch.org/whl/cu124'}
if 'P100' in assigned_gpu_name:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall', f"torch=={P100_STACK['torch']}", f"torchvision=={P100_STACK['torchvision']}", f"xformers=={P100_STACK['xformers']}", f"numpy=={P100_STACK['numpy']}", '--index-url', P100_STACK['index']], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', f"torchao=={P100_STACK['torchao']}"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU assigned. In Kaggle Settings enable a GPU accelerator, then restart the session.')
gpu = torch.cuda.get_device_properties(0)
torch.ones(1, device='cuda').sum().item()
torch.cuda.synchronize()
VISIBLE_GPU_COUNT = torch.cuda.device_count()
print('GPU:', gpu.name)
print('CUDA capability:', f'{gpu.major}.{gpu.minor}')
print('Visible GPUs:', VISIBLE_GPU_COUNT, '(F1-P1 uses device 0 only)')
print('Total VRAM GiB:', round(gpu.total_memory / 1024**3, 2))
print('Initially free VRAM GiB:', round(torch.cuda.mem_get_info()[0] / 1024**3, 2))

## 2. Safe repository setup
This cell clones Musahhih when absent. An existing clean checkout is updated with a fast-forward pull; local work is never deleted or reset.

In [ ]:
from pathlib import Path
import os, subprocess
RUNTIME_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPO = RUNTIME_ROOT / 'Musahhih'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/ALBA7OOTH-Research-Lab/Musahhih.git', str(REPO)], check=True)
else:
    dirty = subprocess.run(['git', '-C', str(REPO), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout
    if dirty:
        print('Existing repository has local changes; leaving it unchanged.')
    else:
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
print('Repository:', REPO)

## 3. Install and record dependencies
This follows the currently supported Unsloth Gemma 3 route. Package versions are recorded in every artifact. If installation changes a version, the later smoke summary—not an assumption—determines compatibility.

In [ ]:
import re
torch_minor = re.match(r'[0-9]+\.[0-9]+', torch.__version__).group(0)
xformers_version = {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2','2.6':'0.0.29.post3'}.get(torch_minor, '0.0.34')
def pip_install(*items):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *items], check=True)
pip_install('sentencepiece', 'protobuf', 'datasets==4.3.0', 'huggingface_hub>=0.34.0', 'hf_transfer')
pip_install('--no-deps', 'unsloth_zoo', 'bitsandbytes', 'accelerate', f'xformers=={xformers_version}', 'peft', 'trl', 'triton', 'unsloth')
pip_install('--no-deps', '--upgrade', 'torchao==0.12.0' if torch_minor == '2.6' else 'torchao>=0.16.0')
pip_install('transformers==4.56.2')
pip_install('--no-deps', 'trl==0.22.2')
print('Installation complete. Restart only if pip explicitly requests it.')

In [ ]:
import importlib.metadata as md, json, platform
packages = ['torch','transformers','unsloth','accelerate','peft','trl','datasets','bitsandbytes']
VERSIONS = {name: md.version(name) for name in packages}
RUNTIME = {'python': platform.python_version(), 'cuda': torch.version.cuda, 'gpu': gpu.name, 'cuda_capability': f'{gpu.major}.{gpu.minor}', 'visible_gpu_count': VISIBLE_GPU_COUNT, 'selected_device': 0, 'p100_stack': P100_STACK if 'P100' in assigned_gpu_name else None, 'packages': VERSIONS}
print(json.dumps(RUNTIME, indent=2))

## 4. Private F1 records
Attach the unchanged licensed QALB archive and verified manifests through a **private Kaggle Dataset**. The staging cell copies only expected files into Git-ignored working paths and never displays their contents. Creating text-bearing records requires institutional/license guidance and both explicit adapter flags. Never publish the Kaggle Dataset.

In [ ]:
# On Kaggle, edit this path to the attached PRIVATE dataset directory.
KAGGLE_PRIVATE_INPUT_DIR = Path('/kaggle/input/musahhih-private-qalb-091')
KAGGLE_PRIVATE_SOURCE_DIR = Path('/kaggle/input/musahhih-private-qalb-source-091')
if Path('/kaggle/input').exists():
    import shutil
    expected_private_files = [
        (KAGGLE_PRIVATE_SOURCE_DIR, 'QALB-0.9.1-Dec03-2021-SharedTasks.zip.bin', Path('data/raw/qalb/QALB-0.9.1-Dec03-2021-SharedTasks.zip')),
        (KAGGLE_PRIVATE_INPUT_DIR, 'qalb_train_selection.jsonl', Path('data/processed/qalb/qalb_train_selection.jsonl')),
        (KAGGLE_PRIVATE_INPUT_DIR, 'qalb_dev_selection.jsonl', Path('data/processed/qalb/qalb_dev_selection.jsonl')),
        (KAGGLE_PRIVATE_INPUT_DIR, 'qalb_manifest_summary.json', Path('data/processed/qalb/qalb_manifest_summary.json')),
        (KAGGLE_PRIVATE_INPUT_DIR, 'LICENSE.txt', Path('data/processed/qalb/f1_p1/LICENSE.txt')),
    ]
    if not KAGGLE_PRIVATE_INPUT_DIR.is_dir() or not KAGGLE_PRIVATE_SOURCE_DIR.is_dir():
        raise RuntimeError('Attach both private QALB Kaggle Datasets before running.')
    for source_dir, filename, destination in expected_private_files:
        matches = list(source_dir.rglob(filename))
        if len(matches) != 1:
            raise RuntimeError(f'Expected exactly one private {filename}; observed {len(matches)}')
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(matches[0], destination)
    print('Expected private archive/manifests staged without displaying corpus text.')
# Run only after the team has the required guidance. This writes solely to ignored data/processed/.
CONFIRM_LICENSE_GUIDANCE = False
if CONFIRM_LICENSE_GUIDANCE:
    subprocess.run([sys.executable, 'scripts/prepare_f1_natural_records.py', '--write-private-records', '--confirm-license-guidance'], check=True)
else:
    print('Private-record creation skipped. Set CONFIRM_LICENSE_GUIDANCE=True only after guidance is documented.')

In [ ]:
from scripts.f1_training_utils import *
TRAIN_PATH = Path('data/processed/qalb/f1_p1/train_records.jsonl')
DEV_PATH = Path('data/processed/qalb/f1_p1/dev_records.jsonl')
TRAIN_INPUT = validate_private_jsonl(TRAIN_PATH, 'train', TRAIN_RECORDS)
DEV_INPUT = validate_private_jsonl(DEV_PATH, 'development', None)
print(json.dumps({'train': TRAIN_INPUT, 'development': DEV_INPUT}, indent=2))
print('Validation metadata contains no corpus text.')

## 5. Load frozen model and attach LoRA
This requires Hugging Face access to the recorded checkpoint. No token is stored in the notebook. The immutable revision, 4-bit load, sequence length, processor, and LoRA targets are recorded. Base weights stay frozen.

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
model, processor = FastModel.from_pretrained(model_name=MODEL_ID, revision=MODEL_REVISION, max_seq_length=MAX_SEQUENCE_LENGTH, load_in_4bit=True)
processor = get_chat_template(processor, chat_template='gemma-3')
model = FastModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0.0, bias='none', target_modules=list(LORA_TARGETS), use_gradient_checkpointing='unsloth', random_state=SEEDS[0])
MODEL_INFO = {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'max_sequence_length': MAX_SEQUENCE_LENGTH, 'load_in_4bit': True, 'lora_targets': LORA_TARGETS}
print(json.dumps(MODEL_INFO, indent=2))

## 6. Format privately and build the completion-only trainer
The cell reads private records without printing them, applies the Gemma 3 chat template once, and masks user/padding tokens. It fails if any example exceeds 1,024 tokens or if masking produces no trainable completion tokens.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
private_data = load_dataset('json', data_files={'train': str(TRAIN_PATH), 'validation': str(DEV_PATH)})
def format_row(row):
    messages = row['prompt'] + row['completion']
    return {'text': processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}
private_data = private_data.map(format_row, remove_columns=private_data['train'].column_names)
lengths = [len(processor(text, add_special_tokens=False).input_ids) for text in private_data['train']['text']]
if max(lengths) > MAX_SEQUENCE_LENGTH:
    raise RuntimeError(f'Frozen sequence limit exceeded: max={max(lengths)}')
OUTPUT_DIR = Path('outputs/F1-P1__gemma3-4b-it__qalb14-dev__s3407__r01')
if OUTPUT_DIR.exists():
    raise RuntimeError('Run directory already exists; never overwrite an experiment ID.')
args = SFTConfig(output_dir=str(OUTPUT_DIR), max_length=MAX_SEQUENCE_LENGTH, dataset_text_field='text', eval_strategy='epoch', save_strategy='epoch', save_total_limit=2, report_to='none', bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(), seed=SEEDS[0], **TRAINING_CONFIG)
trainer = SFTTrainer(model=model, processing_class=processor, train_dataset=private_data['train'], eval_dataset=private_data['validation'], args=args)
trainer = train_on_responses_only(trainer, instruction_part='<start_of_turn>user\n', response_part='<start_of_turn>model\n')
first_labels = trainer.train_dataset[0]['labels']
if not any(label != -100 for label in first_labels):
    raise RuntimeError('Completion masking failed: no trainable assistant tokens.')
LONGEST_INDEX = max(range(len(lengths)), key=lengths.__getitem__)
print({'train_records': len(lengths), 'development_records': len(private_data['validation']), 'max_tokens': max(lengths), 'longest_index': LONGEST_INDEX})

## 7. Deliberate one-batch GPU memory smoke test
Set `RUN_GPU_SMOKE=True` only after reviewing every prior cell. This performs one training step on the longest selected record, saves no adapter, and records only memory/runtime metadata. Passing requires at least 1 GiB headroom.

In [ ]:
RUN_GPU_SMOKE = False
LOAD_PRIOR_SMOKE_SUMMARY = False
SMOKE_SUMMARY_PATH = Path('outputs/f1_p1_smoke_summary.json')
SMOKE_RUNTIME_TAINTED = False
SMOKE_SUMMARY = {'passed': False, 'executed': False, 'contains_corpus_text': False}
if RUN_GPU_SMOKE and LOAD_PRIOR_SMOKE_SUMMARY:
    raise RuntimeError('Choose either a new smoke run or a prior summary, not both.')
if LOAD_PRIOR_SMOKE_SUMMARY:
    SMOKE_SUMMARY = json.loads(SMOKE_SUMMARY_PATH.read_text(encoding='utf-8'))
if RUN_GPU_SMOKE:
    torch.cuda.reset_peak_memory_stats()
    original_dataset = trainer.train_dataset
    original_max_steps = trainer.args.max_steps
    trainer.train_dataset = original_dataset.select([LONGEST_INDEX])
    trainer.args.max_steps = 1
    trainer.args.output_dir = 'outputs/f1_p1_memory_smoke_temporary'
    trainer.train()
    peak = torch.cuda.max_memory_reserved()
    SMOKE_SUMMARY = memory_gate(gpu.total_memory, peak) | {'executed': True, 'model_revision': MODEL_REVISION, 'runtime': RUNTIME}
    trainer.train_dataset = original_dataset
    trainer.args.max_steps = original_max_steps
    SMOKE_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    SMOKE_SUMMARY_PATH.write_text(json.dumps(SMOKE_SUMMARY, indent=2) + '\n', encoding='utf-8')
    SMOKE_RUNTIME_TAINTED = True
    if not SMOKE_SUMMARY['passed']:
        raise RuntimeError('GPU smoke failed the frozen 1 GiB headroom gate.')
print(json.dumps(SMOKE_SUMMARY, indent=2))
if SMOKE_RUNTIME_TAINTED:
    print('Restart the runtime before full training; then set LOAD_PRIOR_SMOKE_SUMMARY=True. This model received the smoke optimizer step.')

## 8. Optional full two-epoch pilot — disabled
Do not enable this cell until the smoke summary passes and the team has reviewed the private-input and artifact paths. The exact confirmation string prevents accidental execution. Epoch checkpoints are later selected only by the frozen development-loss rule; no test set is consulted.

In [ ]:
RUN_FULL_TRAINING = False
FULL_TRAINING_CONFIRMATION_VALUE = ''
if RUN_FULL_TRAINING:
    require_full_training_confirmation(FULL_TRAINING_CONFIRMATION_VALUE, SMOKE_SUMMARY)
    if SMOKE_RUNTIME_TAINTED:
        raise RuntimeError('Smoke-contaminated runtime cannot run F1-P1. Restart and load the prior smoke summary.')
    result = trainer.train()
    evaluations = [item for item in trainer.state.log_history if 'eval_loss' in item and 'epoch' in item]
    if len(evaluations) != 2:
        raise RuntimeError(f'Expected exactly two epoch evaluations, observed {len(evaluations)}')
    first, second = evaluations
    selected = first if first['eval_loss'] <= second['eval_loss'] + 1e-6 else second
    SELECTED_CHECKPOINT = OUTPUT_DIR / f"checkpoint-{int(selected['step'])}"
    if not SELECTED_CHECKPOINT.is_dir():
        raise RuntimeError('Selected epoch checkpoint is missing.')
    selection_summary = {'rule':'lowest development assistant-token loss; ties within 1e-6 choose epoch 1', 'evaluations': evaluations, 'selected_checkpoint': SELECTED_CHECKPOINT.name, 'contains_corpus_text': False}
    (OUTPUT_DIR / 'checkpoint_selection.json').write_text(json.dumps(selection_summary, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(selection_summary, indent=2))
else:
    print('Full F1-P1 training is disabled. No training run was started.')

## 9. Optional private 25-record development smoke
After a valid full run, reload only the selected checkpoint and run the frozen 25-record QALB-development smoke. This is a technical artifact check, not checkpoint selection. Raw responses and licensed prompts remain private. It is disabled by default.

In [ ]:
RUN_PRIVATE_DEV_SMOKE = False
if RUN_PRIVATE_DEV_SMOKE:
    if 'SELECTED_CHECKPOINT' not in globals():
        raise RuntimeError('Run and validate full training/checkpoint selection first.')
    selected_model, selected_processor = FastModel.from_pretrained(model_name=str(SELECTED_CHECKPOINT), max_seq_length=MAX_SEQUENCE_LENGTH, load_in_4bit=True)
    selected_processor = get_chat_template(selected_processor, chat_template='gemma-3')
    FastModel.for_inference(selected_model)
    from scripts.nahw_baseline_utils import parse_model_response
    raw_dev = load_dataset('json', data_files={'development': str(DEV_PATH)})['development']
    order = sorted(range(len(raw_dev)), key=lambda i: __import__('hashlib').sha256(f"F1-P1-dev-smoke|3407|{raw_dev[i]['record_id']}".encode()).hexdigest())[:25]
    private_predictions = []
    for index in order:
        row = raw_dev[index]
        inputs = selected_processor.apply_chat_template(row['prompt'], tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
        output = selected_model.generate(input_ids=inputs, do_sample=False, max_new_tokens=32)
        response = selected_processor.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
        parsed, warnings = parse_model_response(response)
        private_predictions.append({'record_id': row['record_id'], 'raw_response': response, 'parsed_response': parsed, 'parsing_warnings': warnings, 'gold_correction': row['completion'][0]['content']})
    prediction_path = OUTPUT_DIR / 'private_dev_smoke_predictions.jsonl'
    prediction_path.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in private_predictions), encoding='utf-8')
    print({'records': len(private_predictions), 'path': str(prediction_path), 'private': True})
else:
    print('Private development smoke is disabled.')

## 10. Export artifacts optionally
Download only text-free summaries directly. Adapter/checkpoint and private prediction export must follow the team's approved private storage policy. Google Drive is optional; never write credentials into the notebook. No Hugging Face upload occurs automatically.